<a href="https://colab.research.google.com/github/NidhiDevrani/California-Housing-Prices-ML-Project/blob/main/California_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Kaggle

California Housing Prices
Median house prices for California districts derived from the 1990 census.


https://www.kaggle.com/datasets/camnugent/california-housing-prices/code

In [132]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Google collab docs/California Housing Prices.csv')
df.head()

In [134]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

Below cell generates html page for all the info  like shape , duplicates , column ino etc.

In [ ]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport(df)
profile.to_file("report.html")

In [ ]:
# shows only total_bedrooms have some null values

df.isnull().sum()

In [139]:
# because median income is measured in tens of thousands of US Dollars , we are changing back into dollars
df['median_income'] = df['median_income'] * 10000

Predict total_bedrooms values to fill the predicted values on empty total_bedrooms rows

In [140]:
#take a seperate dataframe with only rows where total_bedrooms is not null
df2 = df[df['total_bedrooms'].notna()]

In [ ]:
df2.corr(numeric_only=True)["total_bedrooms"]

In [ ]:
df2.corr(numeric_only=True)["total_bedrooms"]

for col in df2.columns:
    if col != "total_bedrooms":
        sns.scatterplot(x=df2[col], y=df2["total_bedrooms"])
        plt.title(f"{col} vs total_bedrooms")
        plt.show()

It seems like total_bedrooms is heavily correlated with households , so we will do Linear Regression here

In [ ]:
from numpy._core.fromnumeric import shape
array = df2[['households','total_bedrooms']].values
# reshape for scaler
X = array[:,0:1]
Y = array[:,1:]
plt.plot(X,Y,'o')

In [144]:
from sklearn.preprocessing import MinMaxScaler
# create scaler
scaler_X = MinMaxScaler()
scaler_Y = MinMaxScaler()

# scale
X_scaled = scaler_X.fit_transform(X)
Y_scaled = scaler_Y.fit_transform(Y)

In [145]:
# Split into train & test 70-30 %
X_train, X_test, y_train, y_test = train_test_split(X_scaled, Y_scaled, test_size=0.3,random_state=None)

In [146]:
# Create model (Linear Regression using Gradient Descent)
model = SGDRegressor(loss = "huber",max_iter=1000, learning_rate='adaptive', eta0=0.3, alpha=0,verbose = 0,tol=0.001)

In [ ]:
# Function to use model with custom batch_size and epoch

from sklearn.utils import shuffle
batch_size = 1486
epochs = 9
train_Loss = [] #store train cost function value for each epoch
test_Loss = []  #store test cost function value for each epoch
iterations = []

for epoch in range(epochs):

    # shuffle data for each epoch for better fitting
    X_train, y_train = shuffle(X_train, y_train)

    # run all batches at a time
    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        model.partial_fit(X_batch, y_batch)

    # create two variables that are y_test and y_train descaled form
    y_test_descaled = y_test.reshape(-1, 1)
    y_test_descaled = scaler_Y.inverse_transform(y_test_descaled)

    y_train_descaled = y_train.reshape(-1, 1)
    y_train_descaled = scaler_Y.inverse_transform(y_train_descaled)


    # predict for testing data and also descale it for calculating true cost
    y_test_pred = model.predict(X_test)
    y_test_pred = y_test_pred.reshape(-1, 1)
    y_test_pred = scaler_Y.inverse_transform(y_test_pred)

    mse_test = mean_squared_error(y_test_descaled, y_test_pred)

    test_Loss.append(mse_test) # append test cost for each epoch

    # predict for training data and also descale it for calculating true cost
    y_train_pred = model.predict(X_train)
    y_train_pred = y_train_pred.reshape(-1, 1)
    y_train_pred = scaler_Y.inverse_transform(y_train_pred)

    mse_train = mean_squared_error(y_train_descaled, y_train_pred)

    train_Loss.append(mse_train)# append train cost for each epoch

    iterations.append(epoch)


# Plot points show test and training data cost function values with each epoch , based on that we can decide how much we have to train the model
plt.plot(iterations,train_Loss, label='Training Loss', color='blue')
plt.plot(iterations,test_Loss, label='Testing Loss', color='red')
plt.show()


In [148]:
y_test_pred = model.predict(X_test)

In [149]:
# descale x_test , y_test , x_train , x_test , y_test_pred

y_test_pred = y_test_pred.reshape(-1, 1)
y_test_pred = scaler_Y.inverse_transform(y_test_pred)

y_train = scaler_Y.inverse_transform(y_train)

y_test = scaler_Y.inverse_transform(y_test)

X_test = scaler_X.inverse_transform(X_test)

X_train = scaler_X.inverse_transform(X_train)

In [ ]:
# find r2_score for model , mean and median to see if the model is better or not

# MSE and R2_score for model
mse = mean_squared_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)


print("MSE:", mse)
print("R2 Score: " , r2)

# MSE and R2_score for mean

mean_val = y_train.mean()

y_test_pred[:] = mean_val  # by default mean_val will be filled

Mean_store = {}
Mean_count = {}

for i in range(len(X_train)):

  if X_train[i,0] not in Mean_store.keys():
    Mean_store[X_train[i,0]] = y_train[i,0]
    Mean_count[X_train[i,0]] = 1
  else:
    Mean_store[X_train[i,0]] += y_train[i,0]
    Mean_count[X_train[i,0]] +=1


for i in range(len(X_test)):

  if X_test[i,0] in Mean_store.keys():
    y_test_pred[i,0] = Mean_store[X_test[i,0]]/Mean_count[X_test[i,0]]




r2_mean = r2_score(y_test, y_test_pred)
print("MSE (mean): ", mean_squared_error(y_test, y_test_pred))
print("R2 Score (mean): " , r2_mean )

# MSE and R2_score for median

median_val = np.median(y_train)
y_test_pred[:] = median_val

Median_store ={}

for i in range(len(X_train)):

  if X_train[i,0] not in Median_store.keys():
    Median_store[X_train[i,0]] = [y_train[i,0]] # Initialize as a list
  else:
    Median_store[X_train[i,0]].append(y_train[i,0]) # Append to the list


for i in range(len(X_test)):

  if X_test[i,0] in Median_store.keys():
    y_test_pred[i,0] = np.median(Median_store[X_test[i,0]]) # Use np.median on the list



r2_median = r2_score(y_test, y_test_pred)
print("MSE (median): ", mean_squared_error(y_test, y_test_pred))
print("R2 Score (median): " , r2_median )


In [ ]:
# R2 score is much more +ve , so we will use our mode;
# now fill the null values by our trained model

mask = df['total_bedrooms'].isnull()

#median_total_bedrooms_value = df[df["total_bedrooms"].notna()]['total_bedrooms'].median()

#print(median_total_bedrooms_value)
#df.loc[mask, 'total_bedrooms'] = median_total_bedrooms_value

tp = df.loc[mask, ['households']]   # shape (207, 1)
predicted_values = model.predict(tp)

df.loc[mask, 'total_bedrooms'] = predicted_values

# **Data filling ends here**

now the null values are filled and whole data is cleaned , we have to check for model prediction for our main column **median_house_value**

In [152]:
# it is observed that per_household values are more relevent and meaningful than the original ones

df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["population_per_household"] = df["population"] / df["households"]
df["bedrooms_per_household"] = df["total_bedrooms"] / df["households"]

In [ ]:
df[['longitude', 'latitude','housing_median_age','households','median_income','bedrooms_per_household','rooms_per_household'
,'population_per_household','median_house_value']].corr(numeric_only=True)

# it is clear that median_house_value is significantly linearly correlated with median_income only

In [ ]:

for col in df.columns:
    if col != "median_house_value":
        df.plot(kind="scatter", x=col, y="median_house_value", alpha=0.05)
        plt.title(f"{col} vs median_house_value")
        plt.show()

# The plot shows that median_house_value also shows some patterns with longitude and latitude

In [155]:
# divided longitude and latitude into some regions
df["latitude_division"] = None

df.loc[(df["latitude"] >= 32) & (df["latitude"] < 34.5), "latitude_division"] = 4
df.loc[(df["latitude"] >= 34.5) & (df["latitude"] < 36.55), "latitude_division"] = 3
df.loc[(df["latitude"] >= 36.55) & (df["latitude"] < 38.8), "latitude_division"] = 2
df.loc[(df["latitude"] >= 38.8) & (df["latitude"] < 43), "latitude_division"] = 1

df["longitude_division"] = None

df.loc[(df["longitude"] >= -127) & (df["longitude"] < -122.8), "longitude_division"] = 1
df.loc[(df["longitude"] >= -122.8) & (df["longitude"] < -121.8), "longitude_division"] = 2
df.loc[(df["longitude"] >= -121.8) & (df["longitude"] < -120), "longitude_division"] = 3
df.loc[(df["longitude"] >= -120) & (df["longitude"] < -117), "longitude_division"] = 4
df.loc[(df["longitude"] >= -117) & (df["longitude"] < -114), "longitude_division"] = 5

In [ ]:
result = df.groupby(["latitude_division", "longitude_division"]).apply(
    lambda x: (x["median_house_value"] <= 220000).mean() * 100)

table = result.unstack()

print(table)
# Table shows percentage of points on each divided region that have median_income <=22000
# High percentage => poor areas
# NaN => no point(house in that range of longitude + latitude)

In [ ]:
result = df.groupby(["latitude_division", "longitude_division"])["median_house_value"].quantile(0.95)

table = result.unstack()

print(table)

# **Testing Models**

Now we know that longitude , latitude and median_income are the three important columns here

In [158]:
# we can experiment different the combinations to check weightage of features

lst1 = ['longitude', 'latitude','housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
        'rooms_per_household',
       'population_per_household', 'bedrooms_per_household'
        ]

lst2 = ['housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
        'rooms_per_household',
       'population_per_household', 'bedrooms_per_household']

lst3 = ['longitude', 'latitude', 'housing_median_age', 'households', 'median_income',
        'rooms_per_household',
       'population_per_household', 'bedrooms_per_household']

lst4 =  ['housing_median_age', 'households', 'median_income',
        'rooms_per_household',
       'population_per_household', 'bedrooms_per_household']

lst = [lst1,lst2,lst3,lst4]

Linear Regression

```
# This is formatted as code
```



In [ ]:
import copy

from sklearn.model_selection import cross_val_score


def find_answer(features , df , answer):

  model = SGDRegressor(loss = "squared_error", max_iter=1000, learning_rate='adaptive', eta0=0.01, alpha=0,verbose = 0,tol=0.001,random_state=None)
  X = df[features].values
  Y = df[[answer]].values
  #create scaler
  scaler_X = MinMaxScaler()
  scaler_Y = MinMaxScaler()

  # scale
  X_scaled = scaler_X.fit_transform(X)
  Y_scaled = scaler_Y.fit_transform(Y)

  # Split into train & test
  X_train, X_test, y_train, y_test = train_test_split(X_scaled, Y_scaled, test_size=0.2, random_state=None)

  # convert back for model
  y_train = y_train.ravel()

  epochs = 10
  batch_size = 1008

  for epoch in range(epochs):
    X_train, y_train = shuffle(X_train, y_train)

    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]
        model.partial_fit(X_batch, y_batch)

  #predict and descale
  y_test_pred = model.predict(X_test)
  y_test_pred = y_test_pred.reshape(-1, 1)
  y_test_pred = scaler_Y.inverse_transform(y_test_pred)

  y_test = scaler_Y.inverse_transform(y_test)

  mse_test = mean_squared_error(y_test, y_test_pred)
  r2 = r2_score(y_test, y_test_pred)

  print("Linear_Regression : ")
  print("MSE:", mse_test)
  print("RMSE: ", np.sqrt(mse_test))
  print("R2 Score: " , r2)




for i in lst:
    find_answer(i , df , 'median_house_value')
    print("\n")


**XGBoostRegressor**

In [ ]:
!pip install xgboost
!pip install --upgrade xgboost

In [ ]:
from xgboost import XGBRegressor

def find_answer_xgb(features, df, answer):

    X = df[features]
    y = df[answer]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=None)

    model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mse_test = mean_squared_error(y_test, y_pred)

    r2 = r2_score(y_test, y_pred)


    print("xgboost : ")
    print("MSE:", mse_test)
    print("RMSE: ", np.sqrt(mse_test))
    print("R2 Score: " , r2)


for features in lst:
    find_answer_xgb(features, df, 'median_house_value')
    print("\n")

LightGBMRegressor

In [ ]:
from lightgbm import LGBMRegressor

def find_answer_lgbm(features, df, answer):

    X = df[features]
    y = df[answer]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=None)

    model = LGBMRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=-1,
        random_state=42)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)


    mse_test = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)


    print("lgbm : ")
    print("MSE:", mse_test)
    print("RMSE: ", np.sqrt(mse_test))
    print("R2 Score: " , r2)



for features in lst:
    find_answer_lgbm(features, df, 'median_house_value')
    print("\n")

RandomForestRegressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def find_answer_rf(features, df, answer):

    X = df[features]
    y = df[answer]

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=None
    )

    # Model
    model = RandomForestRegressor(
        n_estimators=100,     # reduce trees
        max_depth=None,        # limit depth
        random_state=None,
        n_jobs=-1)

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # MSE
    mse_test = mean_squared_error(y_test, y_pred)

    r2 = r2_score(y_test, y_pred)


    print("random forest : ")
    print("MSE:", mse_test)
    print("RMSE: ", np.sqrt(mse_test))
    print("R2 Score: " , r2)



for features in lst:
    find_answer_rf(features, df, 'median_house_value')
    print("\n")

LightGBM is showing least error , hence it is performing best here